# 1.Setup

In [3]:
import pandas as pd
from ri_chat.config import PROCESSED_DATA_DIR

2026-11-06 10:01:54.521 | INFO     | ri_chat.config:<module>:11 - PROJ_ROOT path is: /home/gabriel_santiago/Estudos/Portfolio/ri_chat


# 2.EDA

In [4]:
df = pd.read_parquet(PROCESSED_DATA_DIR / 'chunks.parquet')

In [8]:
print(df.shape)
df.head()

(6831, 10)


,chunk_id,periodo,ano,trimestre,tipo_doc,page_number,chunk_index,char_count,text,text_with_context
0,1T2024_completas_p2_c0,1T2024,2024,1,completas,2,0,897,#### **Índice** \n\n###### **Análise gerencial...,[Relatório Itaú Unibanco | Período: 1T2024 | D...
1,1T2024_completas_p5_c0,1T2024,2024,1,completas,5,0,141,###### Sumário do **Resultado Gerencial** \n\n...,[Relatório Itaú Unibanco | Período: 1T2024 | D...
2,1T2024_completas_p5_c1,1T2024,2024,1,completas,5,1,997,|**Em R**|**$ milhões (exceto onde indicado)**...,[Relatório Itaú Unibanco | Período: 1T2024 | D...
3,1T2024_completas_p5_c2,1T2024,2024,1,completas,5,2,998,"||Índice de Eficiência (IE) (6)|38,3%|40,3%|39...",[Relatório Itaú Unibanco | Período: 1T2024 | D...
4,1T2024_completas_p5_c3,1T2024,2024,1,completas,5,3,660,|**Bala**|Índice de Basileia Consolidado Prude...,[Relatório Itaú Unibanco | Período: 1T2024 | D...


In [9]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6831 entries, 0 to 6830
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   chunk_id           6831 non-null   str  
 1   periodo            6831 non-null   str  
 2   ano                6831 non-null   int64
 3   trimestre          6831 non-null   int64
 4   tipo_doc           6831 non-null   str  
 5   page_number        6831 non-null   int64
 6   chunk_index        6831 non-null   int64
 7   char_count         6831 non-null   int64
 8   text               6831 non-null   str  
 9   text_with_context  6831 non-null   str  
dtypes: int64(5), str(5)
memory usage: 10.6 MB


In [10]:
print("Todos os chunk_ids são únicos?", df["chunk_id"].is_unique)
print("Total de chunks:", len(df))

Todos os chunk_ids são únicos? True
Total de chunks: 6831


In [11]:
print(df["char_count"].describe())
print(f"\nChunks muito pequenos (< 200 chars): {len(df[df['char_count'] < 200])}")

count    6831.000000
mean      688.647636
std       289.218633
min        20.000000
25%       466.000000
50%       792.000000
75%       939.000000
max      1000.000000
Name: char_count, dtype: float64

Chunks muito pequenos (< 200 chars): 660


In [12]:
chunks_p5 = df.query("periodo == '1T2024' and page_number == 5")

for row in chunks_p5.itertuples():
    print(f"\n{'='*30} {row.chunk_id} ({row.char_count} chars) {'='*30}")
    print(row.text_with_context)


============================== 1T2024_completas_p5_c0 (141 chars) ==============================
[Relatório Itaú Unibanco | Período: 1T2024 | Demonstrações: completas | Página: 5]
###### Sumário do **Resultado Gerencial** 

A seguir, apresentamos os indicadores financeiros do Itaú Unibanco, apurados no final do período.

============================== 1T2024_completas_p5_c1 (997 chars) ==============================
[Relatório Itaú Unibanco | Período: 1T2024 | Demonstrações: completas | Página: 5]
|**Em R**|**$ milhões (exceto onde indicado)**|**1T24**|**4T23**|**1T23**|
|---|---|---|---|---|
||Resultado Recorrente Gerencial|9.771|9.401|8.435|
|**DRE**|Produto Bancário (1)|40.353|40.985|37.450|
||Margem Financeira Gerencial (2)|26.880|27.134|24.692|
||Retorno Recorrente Gerencial sobre o Patrimônio Líquido Médio anualizado -Consolidado (3)|21,9%|21,2%|20,7%|
||Retorno Recorrente Gerencial sobre o Patrimônio Líquido Médio anualizado- Brasil (3)|22,7%|22,2%|21,1%|
|**nho**|Retorno Recor

Notei algumas tabelas separadas entre os chunks, com os títulos das linhas em um chunk e as informações em outro. Para tratar desse problema, posso simplesmente passar todos os chunks correspondentes a uma página ou tentar, de alguma forma, repetir os títulos das tabelas em todos os chunks da determinada tabela.

Cada abordagem possui seu trade-off. No primeiro caso, aumentaria o custo computacional para o caso de passar todos os chunks da página ao modelo, entretanto, é relativamente simples implementar, já está tudo pronto na verdade.

Já na segunda abordagem é mais complexo, porém, mais performático computacionalmente. Entretanto, meu objetivo primeiro será de fazer o projeto funcionar e depois otimizar. Por isso, **seguirei com a abordagem de passar todos os chunks de uma determinada página para o LLM**.